In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (approx_count_distinct,
    col,
    countDistinct,
    sum,
    avg,
    round
)

spark = SparkSession.builder.getOrCreate()

# --------------------------------------------------
# Tables / Checkpoint
# --------------------------------------------------

silver_table = "workspace.default.capstone_silver_sales9"
gold_table = "workspace.default.capstone_gold_sales9"

gold_checkpoint = "/Volumes/workspace/ibm/v10/checkpoints/gold_sales9"

# Remove stale checkpoint if needed
try:
    dbutils.fs.rm(gold_checkpoint, recurse=True)
except Exception:
    pass


# --------------------------------------------------
# Read Silver as Stream
# --------------------------------------------------

silver_df = (
    spark.readStream
         .table(silver_table)
)


# --------------------------------------------------
# Gold Aggregation
# --------------------------------------------------

gold_df = (
    silver_df
    .groupBy(
        "order_date",
        "category",
        "state"
    )
    .agg(
        approx_count_distinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum("gross_amount"), 2).alias("total_gross_sales"),
        round(sum("discount_amount"), 2).alias("total_discount"),
        round(sum("net_amount"), 2).alias("total_net_sales"),
        round(avg("net_amount"), 2).alias("average_order_value")
    )
)


# --------------------------------------------------
# Write Gold
# --------------------------------------------------

gold_query = (
    gold_df.writeStream
           .format("delta")
           .option("checkpointLocation", gold_checkpoint)
           .option("mergeSchema", "true")
           .outputMode("complete")
           .trigger(availableNow=True)
           .toTable(gold_table)
)

gold_query.awaitTermination()

print(f"Gold table loaded successfully: {gold_table}")